# Parameter Optimization

Learn how to find optimal parameters for your trading strategies.

## Topics

1. Grid search optimization
2. Walk-forward analysis
3. Optimization metrics
4. Overfitting prevention
5. Multi-objective optimization

In [ ]:
import backtrader as bt
import pandas as pd
import numpy as np
from datetime import datetime

# Create sample data
dates = pd.date_range('2020-01-01', periods=500, freq='D')
prices = 100 + np.cumsum(np.random.randn(500) * 1.5)

data = pd.DataFrame({
    'open': prices + np.random.randn(500) * 0.5,
    'high': prices + abs(np.random.randn(500) * 1),
    'low': prices - abs(np.random.randn(500) * 1),
    'close': prices,
    'volume': np.random.randint(1000, 10000, 500)
}, index=dates)

data.to_csv('sample_data_opt.csv')
print(f"Created {len(data)} days of sample data")

## 1. Grid Search Optimization

Test all combinations of parameters to find the best.

In [ ]:
class MAStrategy(bt.Strategy):
    """Moving average crossover strategy."""
    
    params = (
        ('fast_period', 10),
        ('slow_period', 30),
        ('printlog', False),
    )
    
    def __init__(self):
        self.fast_ma = bt.indicators.SMA(period=self.p.fast_period)
        self.slow_ma = bt.indicators.SMA(period=self.p.slow_period)
        self.crossover = bt.indicators.CrossOver(self.fast_ma, self.slow_ma)
    
    def next(self):
        if not self.position:
            if self.crossover > 0:
                self.buy()
        else:
            if self.crossover < 0:
                self.close()
    
    def stop(self):
        if self.p.printlog:
            print(f'Fast={self.p.fast_period}, Slow={self.p.slow_period}, '
                  f'Final Value={self.broker.getvalue():.2f}')

# Run optimization
cerebro = bt.Cerebro(optreturn=False)

data_feed = bt.feeds.GenericCSVData(
    dataname='sample_data_opt.csv',
    datetime=0, open=1, high=2, low=3, close=4, volume=5,
    dtformat='%Y-%m-%d'
)
cerebro.adddata(data_feed)

# Add strategy with parameter ranges
cerebro.optstrategy(
    MAStrategy,
    fast_period=range(5, 21, 5),   # 5, 10, 15, 20
    slow_period=range(20, 51, 10), # 20, 30, 40, 50
)

cerebro.broker.setcash(10000.0)
cerebro.broker.setcommission(commission=0.001)

# Add analyzers
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

print("Running grid search optimization...")
opt_results = cerebro.run()

# Analyze results
results_list = []
for result in opt_results:
    strat = result[0]
    sharpe = strat.analyzers.sharpe.get_analysis().get('sharperatio', 0)
    returns = strat.analyzers.returns.get_analysis()['rtot']
    dd = strat.analyzers.drawdown.get_analysis()['max']['drawdown']
    
    results_list.append({
        'fast': strat.p.fast_period,
        'slow': strat.p.slow_period,
        'sharpe': sharpe if sharpe else 0,
        'return': returns,
        'drawdown': dd
    })

# Convert to DataFrame
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('sharpe', ascending=False)

print("\nTop 5 Parameter Combinations:")
print(results_df.head())

## 2. Walk-Forward Analysis

Prevent overfitting by testing on out-of-sample data.

In [ ]:
def walk_forward_optimization(data_path, train_period=250, test_period=50, step=50):
    """Perform walk-forward optimization.
    
    Args:
        data_path: Path to data file
        train_period: Days for training
        test_period: Days for testing
        step: Days to move forward
    """
    # Load full dataset
    full_data = pd.read_csv(data_path, index_col=0, parse_dates=True)
    
    results = []
    start_idx = 0
    
    while start_idx + train_period + test_period <= len(full_data):
        # Split data
        train_end = start_idx + train_period
        test_end = train_end + test_period
        
        train_data = full_data.iloc[start_idx:train_end]
        test_data = full_data.iloc[train_end:test_end]
        
        print(f"\nWindow {len(results)+1}:")
        print(f"  Train: {train_data.index[0]} to {train_data.index[-1]}")
        print(f"  Test:  {test_data.index[0]} to {test_data.index[-1]}")
        
        # Save temporary files
        train_data.to_csv('temp_train.csv')
        test_data.to_csv('temp_test.csv')
        
        # Optimize on training data
        cerebro = bt.Cerebro(optreturn=False)
        train_feed = bt.feeds.GenericCSVData(
            dataname='temp_train.csv',
            datetime=0, open=1, high=2, low=3, close=4, volume=5,
            dtformat='%Y-%m-%d'
        )
        cerebro.adddata(train_feed)
        cerebro.optstrategy(
            MAStrategy,
            fast_period=range(5, 21, 5),
            slow_period=range(20, 51, 10),
        )
        cerebro.broker.setcash(10000.0)
        cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
        
        opt_results = cerebro.run()
        
        # Find best parameters
        best_sharpe = -999
        best_params = None
        for result in opt_results:
            strat = result[0]
            sharpe = strat.analyzers.sharpe.get_analysis().get('sharperatio', 0)
            if sharpe and sharpe > best_sharpe:
                best_sharpe = sharpe
                best_params = (strat.p.fast_period, strat.p.slow_period)
        
        print(f"  Best params: Fast={best_params[0]}, Slow={best_params[1]}")
        
        # Test on out-of-sample data
        cerebro = bt.Cerebro()
        test_feed = bt.feeds.GenericCSVData(
            dataname='temp_test.csv',
            datetime=0, open=1, high=2, low=3, close=4, volume=5,
            dtformat='%Y-%m-%d'
        )
        cerebro.adddata(test_feed)
        cerebro.addstrategy(MAStrategy, 
                          fast_period=best_params[0],
                          slow_period=best_params[1])
        cerebro.broker.setcash(10000.0)
        cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
        
        test_results = cerebro.run()
        test_return = test_results[0].analyzers.returns.get_analysis()['rtot']
        
        print(f"  Test return: {test_return:.2%}")
        
        results.append({
            'window': len(results) + 1,
            'fast': best_params[0],
            'slow': best_params[1],
            'train_sharpe': best_sharpe,
            'test_return': test_return
        })
        
        # Move forward
        start_idx += step
    
    return pd.DataFrame(results)

# Run walk-forward analysis
wf_results = walk_forward_optimization('sample_data_opt.csv')
print("\nWalk-Forward Results:")
print(wf_results)
print(f"\nAverage Test Return: {wf_results['test_return'].mean():.2%}")

## 3. Multi-Objective Optimization

Optimize for multiple metrics simultaneously.

In [ ]:
def multi_objective_score(sharpe, returns, drawdown, weights=(0.5, 0.3, 0.2)):
    """Calculate composite score from multiple metrics.
    
    Args:
        sharpe: Sharpe ratio
        returns: Total returns
        drawdown: Maximum drawdown (negative)
        weights: Weights for (sharpe, returns, drawdown)
    """
    # Normalize metrics (simple approach)
    sharpe_norm = max(0, min(sharpe / 3.0, 1.0))  # Cap at 3.0
    returns_norm = max(0, min(returns / 0.5, 1.0))  # Cap at 50%
    dd_norm = max(0, 1 + drawdown / 50.0)  # Lower drawdown is better
    
    score = (weights[0] * sharpe_norm + 
             weights[1] * returns_norm + 
             weights[2] * dd_norm)
    
    return score

# Run optimization with multi-objective scoring
cerebro = bt.Cerebro(optreturn=False)
cerebro.adddata(data_feed)
cerebro.optstrategy(
    MAStrategy,
    fast_period=range(5, 21, 5),
    slow_period=range(20, 51, 10),
)
cerebro.broker.setcash(10000.0)
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

opt_results = cerebro.run()

# Score each result
scored_results = []
for result in opt_results:
    strat = result[0]
    sharpe = strat.analyzers.sharpe.get_analysis().get('sharperatio', 0) or 0
    returns = strat.analyzers.returns.get_analysis()['rtot']
    dd = strat.analyzers.drawdown.get_analysis()['max']['drawdown']
    
    score = multi_objective_score(sharpe, returns, dd)
    
    scored_results.append({
        'fast': strat.p.fast_period,
        'slow': strat.p.slow_period,
        'sharpe': sharpe,
        'return': returns,
        'drawdown': dd,
        'score': score
    })

scored_df = pd.DataFrame(scored_results)
scored_df = scored_df.sort_values('score', ascending=False)

print("\nTop 5 by Multi-Objective Score:")
print(scored_df.head())

## 4. Overfitting Prevention

Techniques to avoid overfitting.

In [ ]:
# 1. Use fewer parameters
# 2. Require minimum number of trades
# 3. Use walk-forward analysis
# 4. Test on multiple time periods
# 5. Use cross-validation

class RobustStrategy(bt.Strategy):
    """Strategy with overfitting prevention."""
    
    params = (
        ('period', 20),  # Single parameter instead of many
        ('min_trades', 10),  # Minimum trades required
    )
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=self.p.period)
        self.trade_count = 0
    
    def next(self):
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                self.buy()
        else:
            if self.data.close[0] < self.sma[0]:
                self.close()
    
    def notify_trade(self, trade):
        if trade.isclosed:
            self.trade_count += 1
    
    def stop(self):
        # Invalidate results if too few trades
        if self.trade_count < self.p.min_trades:
            print(f"Period {self.p.period}: INVALID (only {self.trade_count} trades)")
        else:
            print(f"Period {self.p.period}: Valid ({self.trade_count} trades)")

# Test with minimum trade filter
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(RobustStrategy, period=20)
cerebro.broker.setcash(10000.0)
cerebro.run()

## 5. Visualization

Visualize optimization results.

In [ ]:
import matplotlib.pyplot as plt

# Create heatmap of results
pivot_table = results_df.pivot(index='slow', columns='fast', values='sharpe')

plt.figure(figsize=(10, 6))
plt.imshow(pivot_table, cmap='RdYlGn', aspect='auto')
plt.colorbar(label='Sharpe Ratio')
plt.xlabel('Fast Period')
plt.ylabel('Slow Period')
plt.title('Parameter Optimization Heatmap')
plt.xticks(range(len(pivot_table.columns)), pivot_table.columns)
plt.yticks(range(len(pivot_table.index)), pivot_table.index)
plt.tight_layout()
plt.savefig('optimization_heatmap.png')
print("Heatmap saved to optimization_heatmap.png")

## Summary

You've learned:
- ✅ Grid search optimization
- ✅ Walk-forward analysis
- ✅ Multi-objective optimization
- ✅ Overfitting prevention
- ✅ Result visualization

## Best Practices

1. **Always use out-of-sample testing**
2. **Require minimum number of trades**
3. **Use walk-forward analysis**
4. **Optimize for multiple metrics**
5. **Keep parameters simple**
6. **Test across different market conditions**

## Next Steps

- **Tutorial 05**: Live Trading with CCXT
- Apply optimization to your own strategies
- Experiment with different metrics